# type_checking

Type checking is not about memorizing `type()` and `isinstance()`. It is about designing code that behaves safely at system boundaries where data is uncertain: APIs, configs, files, CLI input, and pipelines.


## 1. Problem First

Why does this matter in real systems?

- JSON input arrives as untrusted mixed types.
- CLI values are strings until validated.
- Naive type checks reject subclasses or allow dangerous assumptions.


In [ ]:
payload = {"retries": "3"}
print(type(payload["retries"]))


## 2. Minimal Concept

Core tools

- `type(value)` gives the exact runtime type
- `isinstance(value, cls)` checks type compatibility
- `is` checks object identity, not type


## 3. Mental Model

How Python actually behaves

Python is dynamically typed, so names are not locked to types. The object carries the type, not the variable. Good type checking usually happens at system boundaries.


In [ ]:
value = 3
print(type(value))
value = "3"
print(type(value))


## 4. Internal Mechanics

`type(obj)` returns the exact class. `isinstance(obj, cls)` walks inheritance, so it is usually more flexible. This matters because booleans are instances of `int`.


In [ ]:
import dis

print(type(True))
print(isinstance(True, int))
print(type(True) is int)

def accepts_int(value):
    return isinstance(value, int)

dis.dis(accepts_int)


## 5. Edge Cases

Where it breaks

- `type(x) == int` rejects subclasses.
- `isinstance(True, int)` is `True`.
- Over-checking types can fight duck typing.


In [ ]:
print(isinstance(True, int))
print(type(True) == int)

class UserId(int):
    pass

uid = UserId(7)
print(type(uid) == int)
print(isinstance(uid, int))


## 6. Performance Thinking

- Type checks are usually O(1).
- The real cost is architectural: validating too late causes bigger failures.
- Validate once at boundaries, then work with trusted shapes internally.


## 7. Real Use Case

A config loader reads environment variables and JSON fields. Before constructing runtime config, it must verify shape and types clearly.


In [ ]:
config = {"port": 8080, "debug": False}

def validate_config(cfg):
    return isinstance(cfg.get("port"), int) and isinstance(cfg.get("debug"), bool)

print(validate_config(config))


## 8. Anti-Pattern

What beginners do wrong

- Sprinkle `type()` checks everywhere.
- Use exact type equality where `isinstance()` is better.
- Forget that validation should happen close to input boundaries.


In [ ]:
def bad_validate(value):
    return type(value) == int

print(bad_validate(True))
print(bad_validate(3))


## 9. Interview Signals

Questions FAANG asks

- What is the difference between `type()` and `isinstance()`?
- Why is `bool` considered an `int` in Python?
- When should you prefer duck typing over explicit type checks?
- Where in a system should validation happen?


## 10. Exercise (Non-trivial)

Design a validator for job config loaded from JSON. Inputs may contain strings where integers are expected, missing values, explicit booleans, and unknown keys.


In [ ]:
def validate_job_config(config):
    return isinstance(config, dict)

# Task:
# 1. Validate required keys and types.
# 2. Decide whether strings like "3" should be accepted.
# 3. Explain why boundary validation is the right place.
